In [ ]:
%pip install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
import json
from google.colab import userdata

In [3]:
# !cat locations_and_compounds_data.txt blog_data_extracted.txt > locations_compounds_and_blogs_data.txt

In [4]:
# loader = TextLoader("locations_compounds_and_blogs_data.txt")

loader = TextLoader("/kaggle/input/datasets/ahmedayman7/locations-compounds-and-blogs-data-txt/locations_compounds_and_blogs_data.txt")
docs = loader.load()  # Returns list of Document objects

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Max characters per chunk
    chunk_overlap=200,   # Overlap for context
    add_start_index=True # Track original position
)
chunks = splitter.split_documents(docs)

In [7]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [8]:
# Persist to Chroma - best for local dev/production similarity search
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./locations_compounds_and_blogs_db"  # Saves locally
)

In [9]:
!zip -r locations_compounds_and_blogs_db.zip locations_compounds_and_blogs_db

  adding: locations_compounds_and_blogs_db/ (stored 0%)
  adding: locations_compounds_and_blogs_db/1612ea10-9c79-48c3-ae4d-31db7d0a33c9/ (stored 0%)
  adding: locations_compounds_and_blogs_db/1612ea10-9c79-48c3-ae4d-31db7d0a33c9/link_lists.bin (deflated 77%)
  adding: locations_compounds_and_blogs_db/1612ea10-9c79-48c3-ae4d-31db7d0a33c9/length.bin (deflated 95%)
  adding: locations_compounds_and_blogs_db/1612ea10-9c79-48c3-ae4d-31db7d0a33c9/data_level0.bin (deflated 53%)
  adding: locations_compounds_and_blogs_db/1612ea10-9c79-48c3-ae4d-31db7d0a33c9/header.bin (deflated 55%)
  adding: locations_compounds_and_blogs_db/1612ea10-9c79-48c3-ae4d-31db7d0a33c9/index_metadata.pickle (deflated 45%)
  adding: locations_compounds_and_blogs_db/chroma.sqlite3 (deflated 53%)


In [ ]:
from google.colab import files
files.download("locations_and_compounds_db.zip")